# 15. Stacking Ensemble Baseline

**Tujuan:** Training Stacking Ensemble (XGBoost + LightGBM + CatBoost) sebagai base learners
dengan Logistic Regression sebagai meta-learner. Evaluasi performa vs single model.

**Input:** `cleaned_100.pkl`, `experiment_results_03.pkl` (untuk comparison)

**Output:** `stacking_baseline_15.pkl`, model files, PNG visualisasi

In [ ]:
import sys
!{sys.executable} -m pip install lightgbm catboost scikit-learn xgboost -q


import numpy as np
import pandas as pd
import pickle
import os
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
TEST_SIZE = 0.20
DATA_DIR = '../data/'
MODEL_DIR = '../models/'

print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Load cleaned dataset dari notebook 02
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
feature_names = data.get('feature_names', list(range(X.shape[1])))
label_mapping = data.get('label_mapping', None)

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: {len(np.unique(y))}')

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 2. Define Base Learners & Meta-Learner

In [ ]:
# Base Learners
base_learners = {
    'XGBoost': XGBClassifier(
        max_depth=6, n_estimators=100, learning_rate=0.1,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=RANDOM_SEED, verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        max_depth=6, n_estimators=100, learning_rate=0.1,
        random_state=RANDOM_SEED, verbose=-1
    ),
    'CatBoost': CatBoostClassifier(
        depth=6, iterations=100, learning_rate=0.1,
        random_seed=RANDOM_SEED, verbose=0
    )
}

# Meta-Learner
meta_learner = LogisticRegression(
    max_iter=1000, random_state=RANDOM_SEED, multi_class='multinomial'
)

print(f'Base Learners: {list(base_learners.keys())}')
print(f'Meta-Learner: LogisticRegression')

## 3. Stacking — Level 0: Generate Meta-Features via CV

In [ ]:
# Generate out-of-fold predictions (meta-features) using 5-fold CV
n_classes = len(np.unique(y_train))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

meta_train = np.zeros((X_train.shape[0], n_classes * len(base_learners)))
meta_test = np.zeros((X_test.shape[0], n_classes * len(base_learners)))

base_results = {}

print('='*60)
print('  STACKING LEVEL 0: Training Base Learners')
print('='*60)

for idx, (name, model) in enumerate(base_learners.items()):
    print(f'\n  [{idx+1}/{len(base_learners)}] Training {name}...')
    start_time = time.time()
    
    # Out-of-fold predictions for meta-features
    oof_preds = np.zeros((X_train.shape[0], n_classes))
    test_preds = np.zeros((X_test.shape[0], n_classes))
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_tr, y_tr)
        
        oof_preds[val_idx] = model_clone.predict_proba(X_val)
        test_preds += model_clone.predict_proba(X_test) / cv.n_splits
    
    # Store meta-features
    col_start = idx * n_classes
    col_end = (idx + 1) * n_classes
    meta_train[:, col_start:col_end] = oof_preds
    meta_test[:, col_start:col_end] = test_preds
    
    # Evaluate base learner individually
    train_time = time.time() - start_time
    y_pred_base = np.argmax(test_preds, axis=1)
    base_f1 = f1_score(y_test, y_pred_base, average='weighted')
    base_mcc = matthews_corrcoef(y_test, y_pred_base)
    
    base_results[name] = {
        'f1_score': base_f1,
        'mcc': base_mcc,
        'train_time': train_time
    }
    print(f'    F1={base_f1*100:.2f}% | MCC={base_mcc:.4f} | Time={train_time:.1f}s')

print(f'\nMeta-features shape: {meta_train.shape}')

## 4. Stacking — Level 1: Train Meta-Learner

In [ ]:
print('\n' + '='*60)
print('  STACKING LEVEL 1: Training Meta-Learner')
print('='*60)

# Scale meta-features for Logistic Regression
scaler_meta = StandardScaler()
meta_train_scaled = scaler_meta.fit_transform(meta_train)
meta_test_scaled = scaler_meta.transform(meta_test)

# Train meta-learner
start_time = time.time()
meta_learner.fit(meta_train_scaled, y_train)
meta_time = time.time() - start_time

# Predict
y_pred_stack = meta_learner.predict(meta_test_scaled)

# Evaluate Stacking
stack_f1 = f1_score(y_test, y_pred_stack, average='weighted')
stack_mcc = matthews_corrcoef(y_test, y_pred_stack)
stack_acc = accuracy_score(y_test, y_pred_stack)
stack_prec = precision_score(y_test, y_pred_stack, average='weighted')
stack_rec = recall_score(y_test, y_pred_stack, average='weighted')

print(f'\n  Stacking Ensemble Results:')
print(f'    Accuracy:  {stack_acc*100:.2f}%')
print(f'    Precision: {stack_prec*100:.2f}%')
print(f'    Recall:    {stack_rec*100:.2f}%')
print(f'    F1-Score:  {stack_f1*100:.2f}%')
print(f'    MCC:       {stack_mcc:.4f}')
print(f'    Meta-learner time: {meta_time:.2f}s')

## 5. Comparison: Single Models vs Stacking

In [ ]:
# Load previous single-model results for comparison
with open(os.path.join(DATA_DIR, 'experiment_results_03.pkl'), 'rb') as f:
    prev_results = pickle.load(f)

print('\n' + '='*60)
print('  COMPARISON: Single Models vs Stacking Ensemble')
print('='*60)
print(f'{"Model":<20} | {"F1 (%)":<10} | {"MCC":<8}')
print('-'*45)

# Base learners
for name, res in base_results.items():
    print(f'{name:<20} | {res["f1_score"]*100:<10.2f} | {res["mcc"]:<8.4f}')

# Stacking
print(f'{"STACKING ENSEMBLE":<20} | {stack_f1*100:<10.2f} | {stack_mcc:<8.4f}')
print('='*45)

## 6. Train Final Stacking Model (for deployment/next notebooks)

In [ ]:
# Train final base learners on full training data
print('Training final base learners on full training set...')
final_base_models = {}
for name, model in base_learners.items():
    model_clone = model.__class__(**model.get_params())
    model_clone.fit(X_train, y_train)
    final_base_models[name] = model_clone
    print(f'  {name} trained.')

print('Done.')

## 7. Save Results

In [ ]:
stacking_output = {
    # Data splits
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    
    # Meta-features
    'meta_train': meta_train, 'meta_test': meta_test,
    'scaler_meta': scaler_meta,
    
    # Models
    'base_models': final_base_models,
    'meta_learner': meta_learner,
    
    # Results
    'base_results': base_results,
    'stacking_results': {
        'accuracy': stack_acc,
        'precision': stack_prec,
        'recall': stack_rec,
        'f1_score': stack_f1,
        'mcc': stack_mcc
    },
    
    # Config
    'feature_names': feature_names,
    'label_mapping': label_mapping,
    'n_classes': n_classes
}

with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'wb') as f:
    pickle.dump(stacking_output, f)

file_size = os.path.getsize(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl')) / (1024*1024)
print(f'Saved: stacking_baseline_15.pkl ({file_size:.1f} MB)')
print('\nNotebook 15 selesai. Lanjut ke 16 (Feature Ablation).')